In [2]:
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import math
from scratch.linear_algebra import Vector,distance
import random
from collections import Counter
from typing import NamedTuple  

In [3]:
# =================================
# Import DataSet
# =================================
df=pd.read_csv("test_dataSet.csv")
print ('Data Set:')
print (f"Columns\t:{df.shape[1]} \nRows\t:{df.shape[0]}")
df.head()

Data Set:
Columns	:6 
Rows	:100


,No,Usia,BMI,Glukosa,Tekanan_Darah,Outcome
0,1,22,21,85,110,0
1,2,25,23,90,115,0
2,3,28,24,95,118,0
3,4,30,26,100,120,0
4,5,32,27,105,122,0


In [4]:
# ==================================
# Title Columns
# ==================================
print (df.columns)
print ("fixed")
df.columns =df.columns.str.strip().str.lower().str.replace(' ',"_")
print (df.columns)

Index(['No', 'Usia', 'BMI', 'Glukosa', 'Tekanan_Darah', 'Outcome'], dtype='object')
fixed
Index(['no', 'usia', 'bmi', 'glukosa', 'tekanan_darah', 'outcome'], dtype='object')


In [5]:
df.info ()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   no             100 non-null    int64
 1   usia           100 non-null    int64
 2   bmi            100 non-null    int64
 3   glukosa        100 non-null    int64
 4   tekanan_darah  100 non-null    int64
 5   outcome        100 non-null    int64
dtypes: int64(6)
memory usage: 4.8 KB


In [6]:
df.isnull().sum()

no               0
usia             0
bmi              0
glukosa          0
tekanan_darah    0
outcome          0
dtype: int64

In [8]:
# =====================================
# Statistik Central Tedency 
# =====================================
df.describe()

,no,usia,bmi,glukosa,tekanan_darah,outcome
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
mean,50.500000,35.480000,30.040000,120.490000,127.060000,0.500000
std,29.011492,8.373214,6.365818,28.565889,11.730837,0.502519
min,1.000000,21.000000,19.000000,80.000000,105.000000,0.000000
25%,25.750000,28.750000,24.750000,96.750000,118.000000,0.000000
50%,50.500000,35.500000,29.000000,114.500000,125.500000,0.500000
75%,75.250000,42.000000,35.000000,140.500000,136.250000,1.000000
max,100.000000,52.000000,44.000000,195.000000,155.000000,1.000000


In [9]:
df.head (5)

,no,usia,bmi,glukosa,tekanan_darah,outcome
0,1,22,21,85,110,0
1,2,25,23,90,115,0
2,3,28,24,95,118,0
3,4,30,26,100,120,0
4,5,32,27,105,122,0


In [10]:
# rescaling data
keys=['usia','usia','bmi','glukosa','tekanan_darah']
print (keys)

['usia', 'usia', 'bmi', 'glukosa', 'tekanan_darah']


In [11]:
for i in keys:
    min_=df[i].min()
    max_=df[i].max()
    val=[]
    for j in df[i]:
        new_val=(j-min_)/(max_-min_)
        val.append(new_val)
    print (val)
    df[i]=val
    

[np.float64(0.03225806451612903), np.float64(0.12903225806451613), np.float64(0.22580645161290322), np.float64(0.2903225806451613), np.float64(0.3548387096774194), np.float64(0.45161290322580644), np.float64(0.5483870967741935), np.float64(0.6129032258064516), np.float64(0.6774193548387096), np.float64(0.7741935483870968), np.float64(0.06451612903225806), np.float64(0.16129032258064516), np.float64(0.25806451612903225), np.float64(0.3225806451612903), np.float64(0.41935483870967744), np.float64(0.5161290322580645), np.float64(0.5806451612903226), np.float64(0.6451612903225806), np.float64(0.7419354838709677), np.float64(0.8387096774193549), np.float64(0.0967741935483871), np.float64(0.1935483870967742), np.float64(0.2903225806451613), np.float64(0.3870967741935484), np.float64(0.4838709677419355), np.float64(0.5483870967741935), np.float64(0.6129032258064516), np.float64(0.7096774193548387), np.float64(0.8064516129032258), np.float64(0.8709677419354839), np.float64(0.03225806451612903)

In [12]:
# ======================================
# buid model 
# ======================================
# set feature
x =df[df.columns[:-1]]
y= df[df.columns[-1]]

def varible(x,y):
    points =x.values.tolist()
    keys =y.values.tolist()
    return points,keys

points,keys=varible(x,y )


In [13]:
# =================================
# split data test and train
# =================================

def shuffle_index(data:list[int],prob:float):
    data=data[:]
    random.shuffle(data)
    cut=int (len(data)*prob)
    return data[:cut],data[cut:]
    
def split_train_test(xs:list[Vector],ys:list[float],test_pct:float):
    # index 
    idx=[i for i in range(len(xs))]
    train_pct=1 -test_pct
    idx_train,idx_test=shuffle_index(idx,train_pct)
    return (
        # train
        [xs[i] for i in idx_train],
        [ys [i] for i in  idx_train],
        # test 
        [xs[i] for i in idx_test],
        [ys [i] for i in idx_test]
    )

In [14]:
# K-Nearst model with scracth 

# vote fungstion
def vote(labels:list[str]):
    count =Counter(labels)
    winner, winner_val =count.most_common(1)[0]
    # if winner is not 1
    num_winner =len([i for i in count.values() if i == winner_val])
    if num_winner==1:
        return winner 
    else :
        return vote(labels[:-1])
    

In [15]:
class labelPoints(NamedTuple):
    point:Vector
    label:str
    
def k_near(k:int,labelpoint:list[labelPoints],new_point:Vector):
    by_distance=sorted(labelpoint,key=lambda lp: distance(lp.point,new_point))
    label_k=[lp.label for lp in by_distance[:k]]
    return vote(label_k)



In [52]:
points_train,keys_train,points_test,keys_test =split_train_test (points,keys,0.10)

In [53]:
lp_train = [
    labelPoints(p, l)
    for p, l in zip(points_train, keys_train)
]
lp_test =[
    labelPoints(p,l) for p,l in zip(points_test,keys_test)
]


In [54]:
# full varible 
num_corect=0
for test in lp_test:
    predict = k_near(2,lp_train,test.point)
    actual =test.label
    if predict==actual:
        num_corect +=1
    
persentage=num_corect/len(lp_test)
print(f"accuracy :{persentage}")

accuracy :0.9
